# Diagnostic day — H1 / H2 / H3 on Pistonball

Focused read of the **hard OOD matrix** (`reports/rliable_matrix_hard.json`) to decide the next experiment.

| Hypothesis | Claim | Decision rule |
|---|---|---|
| **H1** Env-insensitive | Physics factors barely change returns → context has nothing to condition on | All C.2 axes **FLAT** (incl. gravity) on hard grid |
| **H2** Context redundant | Env is sensitive, but pixels already reveal dynamics → concat ≈ hidden | Non-flat slices **and** concat ≉ better than hidden |
| **H3** Wrong context kind | Need agent-local features, not global physics | Premature until H1 fails |

Regenerate figures anytime (no PettingZoo needed):
```bash
python scripts/diagnostic_day.py
```

In [1]:
from pathlib import Path
import sys

from IPython.display import Image, Markdown, display

REPO = Path("..").resolve()
if not (REPO / "reports").is_dir():
    REPO = Path(".").resolve()
sys.path.insert(0, str(REPO))

from scripts import diagnostic_day as dd
from evaluation.visualize import load_report

REPORT = REPO / "reports" / "rliable_matrix_hard.json"
FIG = REPO / "reports" / "figures" / "diagnostic_day"
FIG.mkdir(parents=True, exist_ok=True)

dd.main(
    dd.Args(
        report_path=str(REPORT),
        standard_report_path=str(REPO / "reports" / "rliable_matrix.json"),
        runs_root=str(REPO / "runs"),
        output_dir=str(FIG),
    )
)
hard = load_report(REPORT)
print("Loaded", REPORT)

OSError: Cannot save file into a non-existent directory: 'reports'

## 1. Methodology filter (what matters)

Ignore transfer plots and exotic metrics for now. Keep: **IQM**, **ID→OOD gap**, **flat/clone flags**, **PoI hidden-varying vs concat-varying**, **seed collapse**.

In [2]:
dash = dd.dashboard_rows(hard["experiments"]).round(3)
display(dash)

print("Env-switch votes:")
for exp in hard["experiments"]:
    env = exp["diagnostics"]["env_switch"]
    print(f"  {exp['label']}: {env['action']} (switch={env['switch_env']})")

for p in hard.get("pairwise_poi") or []:
    if {p["a"], p["b"]} == {"Hidden context (varying)", "Context concat (varying)"}:
        hid_first = p["a"].startswith("Hidden")
        ph = p["probability_a_beats_b"] if hid_first else p["probability_b_beats_a"]
        pc = p["probability_b_beats_a"] if hid_first else p["probability_a_beats_b"]
        print(
            f"\nPoI(hidden-var > concat-var) = {ph['probability_of_improvement']:.3f} "
            f"[{ph['ci_low']:.3f}, {ph['ci_high']:.3f}]"
        )
        print(
            f"PoI(concat-var > hidden-var) = {pc['probability_of_improvement']:.3f} "
            f"[{pc['ci_low']:.3f}, {pc['ci_high']:.3f}]"
        )

NameError: name 'hard' is not defined

## 2. C.2 trapezoid / clone check (H1 gate)

If every axis is **FLAT** (ID−OOD drop < 5) on the **hard** grid including **gravity**, H1 is supported and Multiwalker is the next step.

In [ ]:
slices = dd.slice_rows(hard["experiments"]).round(3)
display(slices)

n_flat = int(slices["flat"].sum())
n_all = len(slices)
n_sens = n_all - n_flat
print(f"Flat slices: {n_flat}/{n_all}; sensitive: {n_sens}")
if n_sens == 0:
    print("→ H1 SUPPORTED: no physics axis is informative under the hard grid.")
else:
    print("→ H1 rejected on at least one axis; inspect sensitive params below.")
    display(slices.loc[~slices["flat"]])

## 3. Decision plots

Pre-rendered under `reports/figures/diagnostic_day/`.

In [ ]:
for name, caption in [
    ("01_iqm_id_ood.png", "IQM and ID/OOD — varying ≫ static; ID≈OOD gaps"),
    ("02_ood_drop_heatmap.png", "Every axis FLAT (H1)"),
    ("03_c2_slices_varying.png", "No trapezoid; mass/elasticity often flat lines"),
    ("04_performance_profiles.png", "Robustness across return thresholds"),
    ("05_concat_minus_hidden.png", "Prefer the median panel; mean is collapse-biased"),
    ("06_seed_health.png", "Collapsed seeds (static especially)"),
]:
    path = FIG / name
    display(Markdown(f"### {caption}"))
    display(Image(filename=str(path)))

## 4. Verdict card

In [ ]:
display(Markdown((FIG / "VERDICT.txt").read_text()))

## How to read this for the thesis

1. **Primary finding on Pistonball:** domain randomization (varying) helps a lot; **oracle physics concat does not** clearly beat hidden; **OOD physics is not a hard generalization stress** (flat C.2).
2. **Do not** invest remaining budget in FiLM / actor–critic / transfer on this env.
3. **Do** run one Multiwalker concat-vs-hidden matrix to test whether H1 is Pistonball-specific.
4. In the write-up, treat Pistonball as a **negative / boundary-condition** pre-study result, not a failed experiment.